In [ ]:
'''
REQUIREMENTS
'''
import os
import sys
from resource import *
import time
import psutil
import csv  #TODO: delete me

def process_memory():
    process = psutil.Process()
    memory_info = process.memory_info()
    memory_consumed = int(memory_info.rss /1024)
    return memory_consumed

def time_wrapper():
    start_time = time.time()
    call_algorithm() # Replace with your algorithm function call
    end_time = time.time()
    time_taken = ( end_time - start_time ) *1000
    return time_taken

In [ ]:
'''
GLOBAL VARIABLES
'''

# Delta and Alphas
alpha = {
  'A': {'A': 0, 'C': 110, 'G': 48, 'T': 94},
  'C': {'A': 110, 'C': 0, 'G': 118, 'T': 48},
  'G': {'A': 48, 'C': 118, 'G': 0, 'T': 110},
  'T': {'A': 94, 'C': 48, 'G': 110, 'T': 0}
}
delta = 30

In [ ]:

# Basic Algorithm (Dynamic Programming)
def sequence_alignment(s1,s2):
  # Base cases: One string is empty, so gap penalty for length of the other string
  dp = [[0 for _ in range(len(s1)+1)] for _ in range(len(s2)+1)]
  for i in range(len(s1)+1):
    dp[i][0] = i * delta
  for j in range(len(s2)+1):
    dp[0][j] = j * delta

  # Recursion
  for i in range(1, len(s1)+1):
    for j in range(1, len(s2)+1):
      dp[i][j] = min(dp[i-1][j] + delta, dp[i][j-1] + delta, dp[i-1][j-1] + alpha[s1[i-1]][s2[j-1]])

  # Pretty print the dp table
  print("Dynamic Programming Table:")
  print("   " + "   ".join([" "] + list(s2)))
  for i, row in enumerate(dp):
    if i == 0:
      print(" " + "   ".join(map(str, row)))
    else:
      print(s1[i-1] + " " + "   ".join(map(str, row)))

  # Export the DP table to a CSV file
  with open('output.csv', 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow([" "] + list(s2))  # Write the header row
    for i, row in enumerate(dp):
      if i == 0:
        csvwriter.writerow([" "] + row)  # Write the first row
      else:
        csvwriter.writerow([s1[i-1]] + row)  # Write subsequent rows
  print("Dynamic Programming Table exported to 'output.csv'")

  print("Value of the optimal solution is ", dp[len(s1)][len(s2)])

  return(dp)

#   alignment_cost = dp[len(s1)][len(s2)]
#   aligned_string_1 = first_string_alignment(s1, s2, dp)
#   aligned_string_2 = second_string_alignment(s1, s2, dp)

#   # save_output(output_file_path, aligned_string_1, aligned_string_2, time_taken, alignment_cost, memory_used)

# Backtrack from dp[n][m] --> seq_array
def string_alignment(s1, s2, dp):
  s1_alignment = []
  s2_alignment = []

  i, j = len(s1), len(s2)
  while i>0 and j>0:
    # Matching / misalignment
    # If the characters match, we take the diagonal value
    # If they don't match, we take the minimum of the three possible moves
    # (up, left, diagonal) and add the corresponding cost
    if i>0 and j>0 and dp[i][j] == dp[i-1][j-1] + alpha[s1[i-1]][s2[j-1]]:
      s1_alignment.append(s1[i-1])
      s2_alignment.append(s2[j-1])
      i-=1
      j-=1
    # Gap penalties (S1)
    elif i>0 and dp[i][j] == dp[i-1][j] + delta:
      s1_alignment.append(s1[i-1])
      s2_alignment.append("_")
      i-=1
    # Gap penalties (S2)
    else:
      s1_alignment.append("_")
      s2_alignment.append(s2[j-1])
      j-=1

  s1_alignment.reverse()
  s2_alignment.reverse()

  print("sAlignments:")
  print("".join(s1_alignment))
  # print("\nS2 Alignment: ")
  print("".join(s2_alignment))






In [ ]:
# Create the output file + remove existing file if it exists
def save_output(output_file_path, aligned_string_1, aligned_string_2, time, alignment_cost, memory_used):
    output_file = os.path.join(output_file_path, "output.txt")
    with open(output_file, "w") as f:
        f.write(f"Alignment Cost: {alignment_cost}\n")
        f.write(f"Aligned String 1: {aligned_string_1}\n")
        f.write(f"Aligned String 2: {aligned_string_2}\n")
        f.write(f"Time Taken (ms): {time}\n")
        f.write(f"Memory Used (KB): {memory_used}\n")
    f.close()
    print("Output file created at:", output_file)


def string_insert(str1, str2, index):
    string1_split = str1[:index]
    string2_split = str1[index:]
    return string1_split + str2 + string2_split


def input_generation(input_file):
    with open(input_file, "r", encoding="utf-8") as f:
        input_text = f.read()

    input_line_sep = input_text.splitlines()
    s1 = input_line_sep[0]
    s2 = ""

    for i in range(1, len(input_line_sep)):
        if s2 == "":
            if input_line_sep[i].isalpha():
                s2 = input_line_sep[i]
            elif input_line_sep[i].isdigit():
                s1 = string_insert(s1, s1, int(input_line_sep[i]) + 1)
        elif input_line_sep[i].isdigit():
            s2 = string_insert(s2, s2, int(input_line_sep[i]) + 1)

    return s1, s2

# DP Algorithm
def main(input_file, output_file_path):
    aligned_string_1 = ""
    aligned_string_2 = ""
    time_taken = 0 # in milliseconds
    memory_used = 0 # in KB
    # alignment_cost = 0

    s1, s2 = input_generation(input_file)
    # print(len(s1), len(s2)) # 32x32
    backtrack = sequence_alignment(s1, s2)
    string_alignment(s1, s2, backtrack)
    # save_output(output_file_path, aligned_string_1, aligned_string_2, time_taken, alignment_cost, memory_used)

    print("s1:", s1)
    print("s2:", s2)

if __name__ == "__main__":
    #TODO: Add command line argument parsing for input file and output directory
    input_file = "input.txt"
    output_file_path = os.getcwd()
    main(input_file, output_file_path)

    # if __name__ == "__main__":
    # if len(sys.argv) != 3:
    #     print("Usage: python script.py <arg1> <arg2>")
    #     sys.exit(1)
#     arg1 = sys.argv[1]
#     arg2 = sys.argv[2]
#     main(arg1, arg2)


